# Segmentation Benchmark — full Colab run

This notebook runs the entire benchmark end-to-end on Colab's GPU. Expected runtime on a T4:
- Data preparation: 15–25 minutes (COCO 2017 downloads are ~20 GB)
- Full training of all 10 models at 512×512 for 25 epochs on 5000 training images: ~10–14 hours
- With `--smoke` (200 train / 40 val / 40 test, 3 epochs): ~40 minutes total

If you are time-constrained, run the smoke test first to verify the pipeline works, then start the full run and let it complete in the background.


## 1. Attach a GPU
**Runtime → Change runtime type → T4 GPU** (or better).

In [ ]:
!nvidia-smi

## 2. Clone the repository

In [ ]:
# From your GitHub after you push, or use a local upload if not yet published
%cd /content
# !git clone https://github.com/<your-user>/Savannah_Shannon_Segmentation_Benchmark.git
%cd /content/Savannah_Shannon_Segmentation_Benchmark

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Prepare the COCO 2017 subset (SEED=42)

In [ ]:
# For a smoke test use small numbers first to verify the pipeline works end-to-end.
# Full protocol (assignment §5): 5000/1000/1000.
!python scripts/prepare_coco_subset.py --train 5000 --val 1000 --test 1000 --yolo-yaml

## 5. (Optional) Quick smoke run — verifies every model works before the full run

In [ ]:
!python run_benchmark.py --task all --model all --smoke

## 6. Full benchmark run

In [ ]:
!python run_benchmark.py --task all --model all

## 7. Regenerate the aggregated CSVs and plots (fast, no training)

In [ ]:
!python run_benchmark.py --aggregate-only
import pandas as pd
pd.read_csv('results/semantic_segmentation_results.csv')

## 8. Inspect a few qualitative comparisons

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
for m in ['unet', 'deeplabv3', 'segformer']:
    files = sorted(os.listdir(f'predictions/{m}'))[:3]
    fig, axes = plt.subplots(1, len(files), figsize=(12, 4))
    for ax, f in zip(axes, files):
        ax.imshow(Image.open(f'predictions/{m}/{f}'))
        ax.set_title(f'{m}/{f}'); ax.axis('off')
    plt.show()

## 9. Persist results back to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Segmentation_Benchmark_Results
!cp -r results plots predictions logs confusion_matrices /content/drive/MyDrive/Segmentation_Benchmark_Results/
print('Results copied to Google Drive.')